## diff ו-cumsum — נגזרת וסכום מצטבר

עכשיו עוברים מ**מבנה נתונים** ל**חשבון אינפיניטסימלי כקריאה לפונקציה** — נגזרת ואינטגרל, כשתי פעולות NumPy על מערך:

- **‎`np.diff(a)`‎** — ההפרש בין כל שני איברים סמוכים: קירוב לנגזרת. מערך באורך $n$ נותן ‎`diff`‎ באורך $n-1$.
- **‎`np.cumsum(a)`‎** — סכום מצטבר: קירוב לאינטגרל (הסכום ההולך וגדל). מערך באורך $n$ נותן ‎`cumsum`‎ **גם** באורך $n$.

```{admonition} 🎥 כאן נכנס סרטון השבוע
:class: seealso

**"נגזרת ואינטגרל: diff, cumsum, trapezoid, quad, ולולאת עדכון"** (כ-6–10 דקות, הקלטת מסך של המחברת עם קול). (הסרטון פרוש על סעיפים 9.8–9.10)

<!-- TODO (למפיק/ה): להחליף תא זה בהטמעת הסרטון בפועל, למשל:
<iframe width="100%" height="400" src="VIDEO_URL_HERE" title="שבוע 9 — נגזרת ואינטגרל: diff, cumsum, trapezoid, quad, ולולאת עדכון" frameborder="0" allowfullscreen></iframe>
-->
```

### דוגמה: מהירות מתוך מיקום, בעזרת ‎`diff`‎

יש לנו את מיקום $x(t)$ של הזריקה, אבל לא נתנו לנו את הנוסחה למהירות — רק את המדידות. ‎`np.diff`‎ מקרב את הנגזרת: $v_x \approx \Delta x/\Delta t$. בגרף שמתחת לתא: הנקודות הן הקירוב, והקו המקווקו הוא הערך המדויק.

In [ ]:
import numpy as np

g = 9.8
theta = np.radians(45)
v0 = 20.0

t_flight = 2 * v0 * np.sin(theta) / g
t = np.linspace(0, t_flight, 60)
x = v0 * np.cos(theta) * t

dx = np.diff(x)
dt = np.diff(t)
vx_approx = dx / dt              # length 59, not 60
t_mid = (t[:-1] + t[1:]) / 2     # the times between the measurements: also 59

vx_exact = v0 * np.cos(theta)    # constant, no air resistance
print(len(t), len(vx_approx), len(t_mid))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
ax.scatter(t_mid, vx_approx, s=10, label="Approximation from diff")
ax.axhline(vx_exact, color="gray", linestyle="--", label="Exact value")
ax.set_ylim(vx_exact - 1, vx_exact + 1)   # the values agree to 1e-11: no offset label
ax.set_xlabel("t (s)")
ax.set_ylabel(r"$v_x$ (m/s)")
ax.legend()
plt.show()

### באג נפוץ: אורך ‎`diff`‎ קצר ב-1

‎`np.diff(x)`‎ על מערך באורך 60 מחזיר מערך באורך **59**, לא 60. ציור שלו מול ‎`t`‎ המקורי, שאורכו 60, נעצר בשגיאה:

```text
ValueError: x and y must have same first dimension, but have shapes (60,) and (59,)
```

גרוע מזה: אם משתמשים בהיסח הדעת ב-‎`t[:-1]`‎, האורכים מתאימים ואין שגיאה, אבל ציר הזמן מוזז בחצי צעד ביחס למה שהמהירות מייצגת. ‎`t_mid`‎ שבדוגמה הוא הפתרון הנכון.

### דוגמה: מרחק כולל שהוטס, בעזרת ‎`cumsum`‎

‎`np.cumsum`‎ על גודל המהירות, כפול צעד הזמן, נותן קירוב למרחק המצטבר שנסע הגוף (לא רק הטווח האופקי הסופי — האורך **לאורך המסלול**).

In [ ]:
y = v0 * np.sin(theta) * t - 0.5 * g * t**2
vx = v0 * np.cos(theta) * np.ones(t.shape)
vy = v0 * np.sin(theta) - g * t
speed = np.sqrt(vx**2 + vy**2)

dt_uniform = t[1] - t[0]
distance_traveled = np.cumsum(speed * dt_uniform)   # same length as t
print(distance_traveled[-1])

In [ ]:
fig, ax = plt.subplots()
ax.plot(t, distance_traveled)
ax.set_xlabel("t (s)")
ax.set_ylabel("Cumulative distance along the path (m)")
plt.show()

### נסו בעצמכם

השלימו את הפונקציה ‎`acceleration`‎, שמקבלת מהירויות ואת זמני המדידה, ומחזירה את התאוצה המקורבת $a\approx\Delta v/\Delta t$ בכל מרווח. בדקו אותה על הדוגמה שבתא, ואחר כך על ‎`vy`‎ מהדוגמה: התוצאה צריכה להיות קרובה ל-$-g$.

In [ ]:
def acceleration(v, t):
    """Approximate acceleration (m/s^2) in each interval: one fewer than v."""
    pass   # replace with your code

v_test = np.array([10.0, 9.0, 8.0])
t_test = np.array([0.0, 0.1, 0.2])
print(acceleration(v_test, t_test))
# expected: [-10. -10.]

`````{admonition} פתרון
:class: dropdown, tip
```python
def acceleration(v, t):
    """Approximate acceleration (m/s^2) in each interval: one fewer than v."""
    return np.diff(v) / np.diff(t)

print(acceleration(v_test, t_test))   # [-10. -10.]
print(acceleration(vy, t)[:3])
```
שלוש מהירויות נותנות שני מרווחים, ולכן שתי תאוצות. ב-‎`vy`‎ של הדוגמה המהירות יורדת קווית, ולכן כל התאוצות שוות ל-$-g$.
`````

### בדקו את עצמכם

שתי השאלות מתייחסות לקוד הבא:

```python
a = np.array([1.0, 4.0, 9.0, 16.0])
print(np.diff(a))      # line A
print(np.cumsum(a))    # line B
```

In [ ]:
import json
from jupyterquiz import display_quiz

example = \
'''
[
    {
        "question": "כמה מספרים תדפיס שורה A?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "3",
                "correct": true,
                "feedback": "נכון: בין 4 ערכים יש 3 הפרשים: 3, 5 ו-7."
            },
            {
                "answer": "4",
                "correct": false,
                "feedback": "לא: diff מחזיר את ההפרשים בין ערכים סמוכים, ויש אחד פחות מהם."
            },
            {
                "answer": "2",
                "correct": false,
                "feedback": "לא: diff מקצר את המערך באחד בלבד."
            },
            {
                "answer": "1",
                "correct": false,
                "feedback": "לא: הפרש בודד בין הערך הראשון לאחרון לא מחושב כאן."
            }
        ]
    },
    {
        "question": "מהו המספר השלישי שתדפיס שורה B?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "14.0",
                "correct": true,
                "feedback": "נכון: הסכום המצטבר עד הערך השלישי הוא 1 ועוד 4 ועוד 9."
            },
            {
                "answer": "9.0",
                "correct": false,
                "feedback": "לא: 9.0 הוא הערך השלישי עצמו; cumsum מחבר את כל הערכים עד אליו."
            },
            {
                "answer": "5.0",
                "correct": false,
                "feedback": "לא: 5.0 הוא הסכום המצטבר השני, 1 ועוד 4."
            },
            {
                "answer": "30.0",
                "correct": false,
                "feedback": "לא: 30.0 הוא הסכום של כל ארבעת הערכים, המספר האחרון."
            }
        ]
    }
]
'''
myquiz = json.loads(example)
display_quiz(myquiz)

### תרגול עצמי

חשבו מהו הטווח האופקי הכולל בעזרת ‎`np.cumsum`‎ על ‎`vx`‎ (כפול ‎`dt_uniform`‎), והשוו לערך התיאורטי $R=v_0^2\sin(2\theta)/g$. מה ההפרש ביניהם, ומה מקורו?

In [ ]:
# x_cumsum = np.cumsum(vx * dt_uniform)
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
x_cumsum = np.cumsum(vx * dt_uniform)
R_theory = v0**2 * np.sin(2 * theta) / g

print("קירוב cumsum:", x_cumsum[-1])
print("תיאוריה:", R_theory)
print("59 מרווחים:", np.cumsum(vx[:-1] * dt_uniform)[-1])
```
הסכום המצטבר יוצא כ-41.5 מטרים, כ-0.7 מטר יותר מהתיאוריה (40.8). הסיבה: ‎`cumsum`‎ מחבר 60 פרוסות של ‎`dt_uniform`‎, אבל בין 60 נקודות זמן יש רק 59 מרווחים, כך שהמעוף "מתארך" בצעד אחד. סכום של 59 פרוסות בלבד נותן את הערך התיאורטי.
`````